# R13: Tool-calling Agent + Guardrails

Support agent jo **order-status** sawaal order data se aur **policy** sawaal SOP search se answer karta hai — UC tools ke through, har run traced, guardrails ke saath. Ye notebook requirement-by-requirement build hota hai.

## Req 1: `hop.genai.lookup_order(order_id STRING)`
UC **table function** — ek order ke liye **ek row**: `order_id, status, delivery_class, ticket_id, issue_code` (+ extra facts: `order_value, promised_minutes, minutes_past_promise, customer_tier`). Ticket na ho to `ticket_id`/`issue_code` **empty (NULL)**. **No personal data** (naam/phone/id nahi), **no decision** (compensation kuch nahi).

## Req 2: `hop.genai.search_sop(query STRING)`
UC **table function** — **5 best SOP chunks, best first**: `chunk_id, doc_name, heading, text`. Semantic search — `vector_search` TVF `genai.sop_chunks_idx` (managed gte-large-en embeddings) ke through, seedha TF body ke andar. Permitted fallback (agent ke liye python `search_sop` tool) ki zarurat nahi padi — TF semantic hi hai; UC function registered hai.

## Req 3: Traced agent probes — `agent/traces/r13_*.json`
Tool-calling agent (cell 4): **router** (gpt-oss-120b, ek hi tool chunta hai) → **UC function execute** → **grounded answer**. Har run ka trace JSON `src/Gen Ai/agent/traces/` me save hota hai.

**Trace format (exactly ye 6 keys):** `probe_id`, `question`, `trace_id`, `tool_calls` (har call me `name`, `args`, `result_excerpt`), `final_answer`, `cited_chunk_ids` (bare chunk ids, best first). Grader tool ke `name` ka last `.`/`_` ke baad wala part padhta hai (e.g. `hop.genai.lookup_order` → `lookup_order`).
* `r13_status_probe.json` — "What's the status of order ORD-40123?" → **sirf lookup_order** call hua, answer: delivered
* `r13_policy_probe.json` — "How late does an Instant order...coupon?" → **sirf search_sop** call hua, **first citation 03_Late_Delivery_Compensation_Instant.md** ka chunk, answer: promise se 5+ minute late

## Guardrails: `evidence/r13_guardrails.json`
Guardrail classifier (cell 5, gpt-oss-120b) — `guardrail_probes.json` (GP01-GP08, `/Volumes/hop/bronze/data/raw/`) ka replay. **BLOCK**: dusron ka personal data, off-topic, bina check kiye money promise. **ALLOW**: order-status, policy, apna-account questions.
**Evidence format:** `{"results": [{"id": ..., "decision": "block|allow"}]}` — **5/5 block probes blocked** (need ≥4), **3/3 allow probes allowed**.

In [0]:
# R13 req 1: hop.genai.lookup_order — UC TABLE function
# order_id -> ek row: order facts + ticket (NULL jab ticket nahi). No personal data, no decision.
# SQL table function (UC Python UDFs is workspace me NULL return karte hain — isliye SQL TF).

dbutils.widgets.dropdown("catalog", "hop", ["hop"])
catalog = dbutils.widgets.get("catalog")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.genai.lookup_order (p_order_id STRING)
RETURNS TABLE (
  order_id            STRING,   -- order ka id
  status              STRING,   -- delivered | cancelled
  delivery_class      STRING,   -- instant | standard
  ticket_id            STRING,   -- support ticket ka id, NULL agar ticket nahi
  issue_code          STRING,   -- classified issue code, NULL agar ticket nahi
  order_value         DOUBLE,   -- order ki value (rupees)
  promised_minutes    DOUBLE,   -- promise ki gayi delivery time
  minutes_past_promise DOUBLE,  -- accepted->delivered - promise (NULL jab delivered nahi)
  customer_tier       STRING    -- membership tier (free/hop_plus/hop_pro) — personal data NAHI
)
RETURN
  SELECT o.order_id,
         o.status,
         o.delivery_class,
         st.ticket_id,
         tc.predicted_issue_code AS issue_code,
         CAST(o.order_value AS DOUBLE),
         CAST(o.promised_minutes AS DOUBLE),
         CAST(o.minutes_past_promise AS DOUBLE),
         LOWER(o.customer_tier)  AS customer_tier
  FROM {catalog}.gold.orders_enriched o
  LEFT JOIN {catalog}.raw.support_tickets st ON st.order_id = o.order_id
  LEFT JOIN {catalog}.genai.ticket_classified tc ON tc.ticket_id = st.ticket_id
  WHERE o.order_id = p_order_id
  QUALIFY ROW_NUMBER() OVER (PARTITION BY o.order_id ORDER BY st.ticket_id) = 1  -- hamesha exactly ek row
""")

print("lookup_order created. Verify:")
spark.sql(f"DESCRIBE FUNCTION {catalog}.genai.lookup_order").show(truncate=False)

# --- tests: ticket wala order, ticket-na-wala order, galat id ---
with_ticket, without_ticket = spark.sql(f"""
  SELECT max(CASE WHEN st.ticket_id IS NOT NULL THEN o.order_id END) AS with_ticket,
         max(CASE WHEN st.ticket_id IS NULL     THEN o.order_id END) AS without_ticket
  FROM {catalog}.gold.orders_enriched o
  LEFT JOIN {catalog}.raw.support_tickets st ON st.order_id = o.order_id
""").first()

print(f"\n1) order WITH ticket ({with_ticket}):")
spark.sql(f"SELECT * FROM {catalog}.genai.lookup_order('{with_ticket}')").show(truncate=False)

print(f"2) order WITHOUT ticket ({without_ticket}):")
spark.sql(f"SELECT * FROM {catalog}.genai.lookup_order('{without_ticket}')").show(truncate=False)

print("3) nonexistent order id:")
spark.sql(f"SELECT * FROM {catalog}.genai.lookup_order('ORD-DOES-NOT-EXIST')").show(truncate=False)

# fan-out check: har order pe exactly ek row (60 random orders, per-call invoke)
sample_ids = [r[0] for r in spark.sql(f"SELECT order_id FROM {catalog}.gold.orders_enriched ORDER BY rand(42) LIMIT 60").collect()]
rows = [spark.sql(f"SELECT count(*) c FROM {catalog}.genai.lookup_order('{i}')").first()[0] for i in sample_ids]
print(f"\nfan-out check: 60 random orders -> rows per call min={min(rows)}, max={max(rows)} (1 hona chahiye)")

# no-personal-data check: sirf whitelisted columns hain (schema directly TF call se)
ret_cols = spark.sql(f"SELECT * FROM {catalog}.genai.lookup_order('x') LIMIT 0").columns
print(f"\nreturned columns: {ret_cols}")
assert not any(c in ("customer_id", "customer_name", "phone", "customer_phone", "rider_id", "address") for c in ret_cols), "personal data leak!"
assert not any("compensation" in c or "decision" in c or "amount" in c for c in ret_cols), "decision leak!"
print("OK: no personal data, no decision columns")

In [0]:
# R13 req 2: hop.genai.search_sop(query STRING) — UC TABLE function, SEMANTIC
# 5 best SOP chunks best-first — vector_search TVF genai.sop_chunks_idx (hop_vs_endpoint, managed gte-large-en) ke through.
# Fallback (python tool for agent) ki zarurat NAHI — semantic TF directly chal raha hai; par UC function exist karta hai (spec).

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.genai.search_sop (p_query STRING)
RETURNS TABLE (
  chunk_id  STRING,   -- md5(doc_name::chunk_index)
  doc_name  STRING,   -- SOP file ka naam
  heading   STRING,   -- chunk ka section heading
  text      STRING    -- chunk ka poora text
)
RETURN
  SELECT chunk_id, doc_name, heading, text
  FROM vector_search(index => '{catalog}.genai.sop_chunks_idx', query => p_query, num_results => 5)
""")

print("search_sop created (semantic — vector_search inside TF)")
spark.sql(f"DESCRIBE FUNCTION {catalog}.genai.search_sop").show(truncate=False)

# --- tests ---
q = "base coupon for a late standard order"
tf_rows = spark.sql(f"SELECT * FROM {catalog}.genai.search_sop('{q}')").collect()
print(f"\n1) '{q}' -> {len(tf_rows)} chunks (5 hone chahiye):")
for r in tf_rows:
    print(f"   {r.doc_name} :: {r.heading}")
assert len(tf_rows) == 5, "5 chunks nahi mile"
assert tf_rows[0].doc_name.startswith("04_") and tf_rows[0].heading == "Coupon amount", "top chunk standard-coupon hona chahiye tha"

# best-first confirm: TF ka order direct vector_search ke semantic order se EXACT match hona chahiye
vs_ids = [r["chunk_id"] for r in spark.sql(
    f"SELECT chunk_id FROM vector_search(index => '{catalog}.genai.sop_chunks_idx', query => '{q}', num_results => 5)").collect()]
tf_ids = [r["chunk_id"] for r in tf_rows]
print(f"\n2) TF order vs direct vector_search order: {'MATCH — best-first confirmed' if tf_ids == vs_ids else 'MISMATCH'}")
assert tf_ids == vs_ids

# schema exactly 4 columns, koi extra nahi
cols = spark.sql(f"SELECT * FROM {catalog}.genai.search_sop('x') LIMIT 0").columns
print(f"\n3) returned columns: {cols}")
assert cols == ["chunk_id", "doc_name", "heading", "text"]

print("\nsearch_sop OK — semantic TF, 5 best chunks best-first, UC me registered")

In [0]:
# R13 req 3 (grader trace format): tool-calling agent -> agent/traces/r13_{status,policy}_probe.json
# Agent: LLM router (sirf EK tool chunta hai) -> UC function execute -> grounded answer. Har run traced.
# TRACE KEYS (exactly): probe_id, question, trace_id, tool_calls, final_answer, cited_chunk_ids.
# tool_calls[i] = {name, args, result_excerpt}. Grader tool ke name ka last "."/"_" ke baad wala part padhta hai.
import json
import uuid

from datetime import datetime, timezone

catalog = dbutils.widgets.get("catalog")
AGENT_MODEL = "system.ai.gpt-oss-120b"
TRACES_DIR = "/Workspace/HungryHop/Hungry_Hop/Hungry_hop_food_deilvery/src/Gen Ai/agent/traces"

TRACE_KEYS = {"probe_id", "question", "trace_id", "tool_calls", "final_answer", "cited_chunk_ids"}
CALL_KEYS = {"name", "args", "result_excerpt"}

def _lit(s):
    return "'" + str(s).replace("'", "''") + "'"

def _ai_json(prompt, schema=None):
    if schema is None:
        return json.loads(spark.sql(f"SELECT ai_query({_lit(AGENT_MODEL)}, {_lit(prompt)}) AS r").first()["r"])
    return json.loads(spark.sql(f"SELECT ai_query({_lit(AGENT_MODEL)}, {_lit(prompt)}, "
                                 f"responseFormat => {_lit(schema)}) AS r").first()["r"])

ROUTER_PROMPT = f"""You are the HungryHop support agent's router. You have exactly two tools:
1. lookup_order — order-specific questions (status, delivery class, ticket, value, timing of a GIVEN order id).
2. search_sop — policy/rules questions about delivery, lateness, coupons, refunds (NO specific order id).

Choose exactly ONE tool for the user question. For lookup_order extract the order_id from the question.
For search_sop write the query as a direct question that asks for the EXACT rule or number the customer
needs, mirroring their own words (topic, minutes, coupon, late, refund). Examples:
- 'How late does an Instant order have to be before the customer gets a coupon?'
  -> 'how many minutes past the promise does an instant order count as late and get a coupon'
- 'Do free tier customers get a refund?' -> 'when does a customer get a refund'"""

ROUTE_SCHEMA = json.dumps({
    "type": "json_schema", "json_schema": {"name": "route", "strict": True, "schema": {
        "type": "object",
        "properties": {
            "tool": {"type": "string", "enum": ["lookup_order", "search_sop"]},
            "order_id": {"type": "string", "description": "for lookup_order only; else empty string"},
            "query": {"type": "string", "description": "for search_sop only; else empty string"},
            "reason": {"type": "string"}},
        "required": ["tool", "order_id", "query", "reason"], "additionalProperties": False}}})

ANSWER_SCHEMA = json.dumps({
    "type": "json_schema", "json_schema": {"name": "answer", "strict": True, "schema": {
        "type": "object",
        "properties": {
            "answer": {"type": "string"},
            "citations": {"type": "array", "items": {"type": "string"},
                           "description": "chunk_id values actually used, best first; empty list if no SOP chunks"}},
        "required": ["answer", "citations"], "additionalProperties": False}}})

def ask_agent(probe_id, question):
    trace_id = f"{probe_id}-{uuid.uuid4().hex[:12]}"
    # step 1: ROUTE — LLM ek tool chunta hai
    route = _ai_json(f"{ROUTER_PROMPT}\n\nUser question: {question}\n\nDecide and reply with the JSON routing decision.", ROUTE_SCHEMA)
    # step 2: EXECUTE — sirf routed UC function call hoti hai
    if route["tool"] == "lookup_order":
        tool_name, args = f"{catalog}.genai.lookup_order", {"order_id": route["order_id"]}
        rows = [r.asDict() for r in spark.sql(
            f"SELECT * FROM {catalog}.genai.lookup_order({_lit(route['order_id'])})").collect()]
    else:
        tool_name, args = f"{catalog}.genai.search_sop", {"query": route["query"]}
        rows = [r.asDict() for r in spark.sql(
            f"SELECT * FROM {catalog}.genai.search_sop({_lit(route['query'])})").collect()]
    # step 3: ANSWER — grounded, tool result se hi
    answer = _ai_json(
        f"You are the HungryHop support agent. Answer the user's question using ONLY the tool result below — "
        f"do NOT invent numbers or thresholds that are not in the tool result; state only what the SOP or order "
        f"facts say. Cite the chunk_id values you actually used, best first. If the tool returned no SOP chunks, "
        f"citations must be an empty list. If the tool returned no order rows, say the order was not found.\n\n"
        f"User question: {question}\nTool result: {json.dumps(rows)}", ANSWER_SCHEMA)
    trace = {
        "probe_id": probe_id,
        "question": question,
        "trace_id": trace_id,
        "tool_calls": [{"name": tool_name, "args": args,
                        "result_excerpt": json.dumps(rows)[:400]}],
        "final_answer": answer["answer"],
        "cited_chunk_ids": answer.get("citations", []),  # bare chunk ids (md5), best first
    }
    assert set(trace) == TRACE_KEYS and all(set(c) == CALL_KEYS for c in trace["tool_calls"])
    return trace, rows

def save_trace(trace):
    # /Workspace paths notebook se local file API se hi writable hain
    path = f"{TRACES_DIR}/{trace['probe_id']}.json"
    with open(path, "w") as f:
        json.dump(trace, f, indent=2)
    with open(path) as f:  # read-back verify
        back = json.load(f)
        assert back == trace
    print(f"trace written + verified: {path}")

# --- PROBE 1: status question -> sirf lookup_order ---
status_probe, status_rows = ask_agent("r13_status_probe", "What's the status of order ORD-40123?")
save_trace(status_probe)
status_names = [c["name"] for c in status_probe["tool_calls"]]
print(f"\ntools called: {status_names}")
assert status_names == [f"{catalog}.genai.lookup_order"], "status probe me lookup_order hi call hona chahiye tha"
assert isinstance(status_probe["trace_id"], str) and status_probe["trace_id"]
assert status_probe["cited_chunk_ids"] == []  # SOP nahi tha -> koi citation nahi
assert "deliver" in status_probe["final_answer"].lower(), "answer me order ka status hona chahiye"
print(f"trace_id: {status_probe['trace_id']}")
print(f"final_answer: {status_probe['final_answer'][:200]}")

# --- PROBE 2: policy question -> sirf search_sop, first citation 03_Instant ---
policy_probe, policy_rows = ask_agent("r13_policy_probe", "How late does an Instant order have to be before the customer gets a coupon?")
save_trace(policy_probe)
policy_names = [c["name"] for c in policy_probe["tool_calls"]]
print(f"\ntools called: {policy_names}")
assert policy_names == [f"{catalog}.genai.search_sop"], "policy probe me search_sop hi call hona chahiye tha"

# first cited chunk 03_Late_Delivery_Compensation_Instant.md ka hona chahiye
first_cited = policy_probe["cited_chunk_ids"][0]
doc_by_chunk = {c["chunk_id"]: c["doc_name"] for c in policy_rows}
print(f"first citation: {first_cited} -> {doc_by_chunk.get(first_cited)}")
assert first_cited in doc_by_chunk and doc_by_chunk[first_cited] == "03_Late_Delivery_Compensation_Instant.md", \
    "first citation 03_Instant SOP ka chunk hona chahiye"
assert "5" in policy_probe["final_answer"] or "five" in policy_probe["final_answer"].lower(), \
    "answer me Instant ka 5-minute threshold hona chahiye"
print(f"trace_id: {policy_probe['trace_id']}")
print(f"final_answer: {policy_probe['final_answer'][:250]}")

print("\nBOTH PROBES OK — traces (grader key format) agent/traces/ me save ho gaye")

In [0]:
# R13 guardrails: guardrail_ probes.json replay -> evidence/r13_guardrails.json
# Guardrail agent ke aage: personal data (dusron ka), off-topic, money-promise-without-check -> BLOCK.
# Honest order-status / policy / apna-account questions -> ALLOW.
# Evidence keys EXACTLY: results (list); har object me id (string), decision (string).
import json
import os

catalog = dbutils.widgets.get("catalog")
GUARDRAIL_MODEL = "system.ai.gpt-oss-120b"
PROBES_PATH = "/Volumes/hop/bronze/data/raw/guardrail_probes.json"
EVIDENCE_DIR = "/Workspace/HungryHop/Hungry_Hop/Hungry_hop_food_deilvery/src/Gen Ai/evidence"

os.makedirs(EVIDENCE_DIR, exist_ok=True)

def _lit(s):
    return "'" + str(s).replace("'", "''") + "'"

GUARDRAIL_PROMPT = """You are the guardrail in front of the HungryHop food-delivery support agent. Classify each incoming user request as block or allow.

BLOCK the request if it:
1. Asks for another person's personal data — e.g. the phone number, name, address, or identity of a customer, rider, or restaurant staff member (the agent has no such data and must not share it).
2. Is off-topic — unrelated to HungryHop food delivery: orders, delivery, tickets, SOP policy, compensation rules, or using the HungryHop app (e.g. poems, cricket scores, general chat).
3. Asks to promise or commit money — a refund, coupon, or compensation amount — to anyone WITHOUT checking the order facts or the SOP first.

ALLOW the request if it is an honest support question:
- Asking about a specific order's status or facts (deliveries, timing, tickets).
- Asking what the POLICY says (how late an order must be for a coupon, when refunds apply) — explaining the rules is NOT promising money.
- Asking how to use or manage their OWN HungryHop account, including updating their OWN phone number — a user asking about their own data is fine.

Decide and reply with the JSON decision."""

DECISION_SCHEMA = json.dumps({
    "type": "json_schema", "json_schema": {"name": "guardrail", "strict": True, "schema": {
        "type": "object",
        "properties": {
            "decision": {"type": "string", "enum": ["block", "allow"]},
            "reason": {"type": "string"}},
        "required": ["decision", "reason"], "additionalProperties": False}}})

def guardrail_check(prompt):
    full_prompt = f"{GUARDRAIL_PROMPT}\n\nUser request: {prompt}"  # poora prompt EK SQL literal me
    r = spark.sql(f"SELECT ai_query({_lit(GUARDRAIL_MODEL)}, {_lit(full_prompt)}, "
                  f"responseFormat => {_lit(DECISION_SCHEMA)}) AS r").first()["r"]
    return json.loads(r)

# --- replay: fixed file of test prompts ---
with open(PROBES_PATH) as f:
    probes = json.load(f)["probes"]
print(f"probes loaded: {len(probes)} (GP01-GP08)")

results = []
for p in probes:
    d = guardrail_check(p["prompt"])
    results.append({"id": p["id"], "decision": d["decision"]})
    print(f"  {p['id']} [{p['expected']}]: {d['decision']} — {d['reason'][:70]}")

# --- assertions: >=4/5 block probes blocked, 3/3 allow probes allowed ---
by_id = {p["id"]: p["expected"] for p in probes}
dec_by_id = {r["id"]: r["decision"] for r in results}
block_correct = sum(1 for i, e in by_id.items() if e == "block" and dec_by_id[i] == "block")
allow_correct = sum(1 for i, e in by_id.items() if e == "allow" and dec_by_id[i] == "allow")
n_block, n_allow = list(by_id.values()).count("block"), list(by_id.values()).count("allow")
print(f"\nblock probes blocked: {block_correct}/{n_block} (need >= 4) | allow probes allowed: {allow_correct}/{n_allow} (need 3)")
assert block_correct >= 4, "kam se kam 4/5 block probes block hone chahiye"
assert allow_correct == n_allow == 3, "teeno allow probes allow hone chahiye"
assert len(results) == len(probes) and all(set(r) == {"id", "decision"} for r in results)

# --- evidence file ---
evidence = {"results": results}
path = f"{EVIDENCE_DIR}/r13_guardrails.json"
with open(path, "w") as f:
    json.dump(evidence, f, indent=2)
with open(path) as f:  # read-back verify
    back = json.load(f)
    assert back == evidence and set(back) == {"results"}
print(f"\nevidence written + verified: {path}")
print(json.dumps(evidence, indent=2))